In [0]:
SELECT
    product_id,
    scraped_date,
    ROUND(AVG(COALESCE(cash_price, list_price)), 2) AS avg_price
FROM
    products.silver_products
GROUP BY
    1,2
ORDER BY
    scraped_date, RIGHT(product_id, 2) DESC;

SELECT 
    product_id,
    scraped_date,
    is_valid_price,
    MIN(COALESCE(cash_price, list_price)) AS min_price,
    MAX(COALESCE(cash_price, list_price)) AS max_price,
    ROUND(((MAX(COALESCE(cash_price, list_price)) - MIN(COALESCE(cash_price, list_price))) / MIN(COALESCE(cash_price, list_price))) * 100, 2) AS price_gap_pct
FROM products.silver_products
WHERE is_available = TRUE
GROUP BY 1, 2, 3
HAVING price_gap_pct > 10 -- Solo nos interesan brechas mayores al 10%
ORDER BY price_gap_pct DESC;

WITH market_stats AS (
  SELECT product_id, scraped_date, min_market_price, avg_market_price
  FROM workspace.products.gold_daily_market_prices
)
SELECT 
    s.scraped_date,
    s.retailer,
    s.product_id,
    COALESCE(s.cash_price, s.list_price) AS retailer_price,
    m.avg_market_price,
    -- Cuánto porcentaje está por debajo del promedio de mercado
    ROUND(((m.avg_market_price - COALESCE(s.cash_price, s.list_price)) / m.avg_market_price) * 100, 2) AS market_discount_edge_pct
FROM workspace.products.silver_products s
INNER JOIN market_stats m 
  ON s.product_id = m.product_id AND s.scraped_date = m.scraped_date
WHERE COALESCE(s.cash_price, s.list_price) = m.min_market_price
  AND ROUND(((m.avg_market_price - COALESCE(s.cash_price, s.list_price)) / m.avg_market_price) * 100, 2) > 15.0;



--¿Qué tan caro o barato estoy respecto al promedio del mercado hoy en mi categoría?
SELECT 
    s.scraped_date,
    s.product_id,
    s.brand,
    c.main_category,
    c.sub_category,
    MIN(COALESCE(s.cash_price, s.list_price)) AS min_market_price,
    MAX(COALESCE(s.cash_price, s.list_price)) AS max_market_price,
    ROUND(AVG(COALESCE(s.cash_price, s.list_price)), 2) AS avg_market_price,
    COUNT(DISTINCT s.retailer) AS active_competitors
FROM 
    workspace.products.silver_products s
INNER JOIN 
    workspace.products.catalog c ON s.product_id = c.product_id
WHERE 
    s.is_valid_price = TRUE 
    AND 
    s.is_available = TRUE
GROUP BY 1, 2, 3, 4, 5;